<a href="https://colab.research.google.com/github/shammintithi/Predicting-Breast-Cancer-Stages-Through-Integrative-Machine-Learning-and-Explainable-AI-Approaches/blob/main/Breast_Cancer_Stage_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Dataset Merge**

In [ ]:
# Import Libraries

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.preprocessing import (LabelEncoder, OneHotEncoder, MinMaxScaler, StandardScaler, LabelBinarizer)
from sklearn.feature_selection import ( VarianceThreshold, f_classif, chi2, SelectKBest, RFE)
from sklearn.model_selection import ( train_test_split, GridSearchCV)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier, AdaBoostClassifier, GradientBoostingClassifier, VotingClassifier)
import xgboost as xgb
import lightgbm as lgb
from imblearn.over_sampling import SMOTENC
from sklearn.metrics import ( accuracy_score, confusion_matrix, classification_report, precision_score, recall_score, f1_score, ConfusionMatrixDisplay, roc_curve, auc, precision_recall_curve, average_precision_score)
from sklearn.inspection import permutation_importance
import shap

In [ ]:
# Cell 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
!ls "/content/drive/MyDrive/Breast_Cancer/Kaggle"
!ls "/content/drive/MyDrive/Breast_Cancer/Metabric"
!ls "/content/drive/MyDrive/Breast_Cancer/SEER"
kaggle_path   = "/content/drive/MyDrive/Breast_Cancer/Kaggle/kaggle.csv"
metabric_patient_path = "/content/drive/MyDrive/Breast_Cancer/Metabric/data_clinical_patient.txt"
metabric_sample_path  = "/content/drive/MyDrive/Breast_Cancer/Metabric/data_clinical_sample.txt"
seer_path     = "/content/drive/MyDrive/Breast_Cancer/SEER/SEER.csv"

In [ ]:
# Cell 3: Load & preprocess of Kaggle Dataset

kaggle = pd.read_csv(kaggle_path)
kaggle = kaggle.rename(columns={
    "Age": "AGE",
    "Tumor Size (cm)": "TUMOR_SIZE_CM",
    "Inv-Nodes": "NODES",
    "Menopause": "MENOPAUSE",
    "Metastasis": "METASTASIS",
    "Breast": "LATERALITY",
    "Breast Quadrant": "QUADRANT",
    "History": "FAMILY_HISTORY",
    "Survival Months": "SURVIVAL_MONTHS",
    "Status": "STATUS",
    "NPI": "NPI"
})

if "Diagnosis Result" in kaggle.columns:
    kaggle = kaggle.drop(columns=["Diagnosis Result"])

if "TUMOR_SIZE_CM" in kaggle.columns:
    kaggle["TUMOR_SIZE_CM"] = pd.to_numeric(kaggle["TUMOR_SIZE_CM"], errors="coerce")
    kaggle["TUMOR_SIZE"] = kaggle["TUMOR_SIZE_CM"] * 10
else:
    kaggle["TUMOR_SIZE"] = None

kaggle = kaggle.drop(columns=["TUMOR_SIZE_CM"], errors="ignore")

if "NODES" in kaggle.columns:
    kaggle["NODES"] = (
        kaggle["NODES"]
        .astype(str)
        .str.extract(r"(\d+)")[0]
    )
    kaggle["NODES"] = pd.to_numeric(kaggle["NODES"], errors="coerce")

kaggle["CANCER_STAGE"] = None

for col in [
    "GRADE", "ER_STATUS", "PR_STATUS",
    "HER2_STATUS", "CELLULARITY", "METASTASIS"
]:
    if col not in kaggle.columns:
        kaggle[col] = None

for col in ["AGE", "SURVIVAL_MONTHS", "NPI"]:
    kaggle[col] = pd.to_numeric(kaggle.get(col), errors="coerce")

if "STATUS" in kaggle.columns:
    kaggle["STATUS"] = (
        kaggle["STATUS"]
        .astype(str)
        .str.strip()
        .str.upper()
    )
else:
    kaggle["STATUS"] = None

kaggle["SOURCE"] = "KAGGLE"

final_cols = [
    "AGE", "TUMOR_SIZE", "NODES", "GRADE", "MENOPAUSE",
    "ER_STATUS", "PR_STATUS", "HER2_STATUS", "CELLULARITY",
    "METASTASIS", "SURVIVAL_MONTHS", "STATUS", "NPI",
    "SOURCE", "CANCER_STAGE"
]

for col in final_cols:
    if col not in kaggle.columns:
        kaggle[col] = None
kaggle_final = kaggle[final_cols]
print("Kaggle dataset loaded.")
print(kaggle_final.head())


In [ ]:
# Cell 4: Load & preprocess METABRIC Dataset

meta_patient = pd.read_csv(metabric_patient_path, sep="\t", comment="#")
meta_sample  = pd.read_csv(metabric_sample_path, sep="\t", comment="#")

metabric = pd.merge(
    meta_patient,
    meta_sample,
    on="PATIENT_ID",
    how="inner"
)

metabric = metabric.rename(columns={
    "AGE_AT_DIAGNOSIS": "AGE",
    "TUMOR_SIZE": "TUMOR_SIZE",
    "LYMPH_NODES_EXAMINED_POSITIVE": "NODES",
    "GRADE": "GRADE",
    "TUMOR_STAGE": "CANCER_STAGE",
    "ER_STATUS": "ER_STATUS",
    "PR_STATUS": "PR_STATUS",
    "HER2_STATUS": "HER2_STATUS",
    "OS_MONTHS": "SURVIVAL_MONTHS",
    "OS_STATUS": "STATUS",
    "INFERRED_MENOPAUSAL_STATE": "MENOPAUSE",
    "CELLULARITY": "CELLULARITY",
    "NPI": "NPI"
})

metabric["AGE"] = pd.to_numeric(metabric["AGE"], errors="coerce")
metabric["NODES"] = pd.to_numeric(metabric["NODES"], errors="coerce")
metabric["TUMOR_SIZE"] = pd.to_numeric(metabric["TUMOR_SIZE"], errors="coerce")

metabric["MENOPAUSE"] = (
    metabric["MENOPAUSE"]
    .astype(str)
    .str.strip()
    .str.title()
)
metabric["MENOPAUSE"] = metabric["MENOPAUSE"].where(
    metabric["MENOPAUSE"].isin(["Pre", "Post"]),
    "Unknown"
)

if "STATUS" in metabric.columns:
    metabric["STATUS"] = (
        metabric["STATUS"]
        .astype(str)
        .str.strip()
        .str.upper()
        .str.split(":")
        .str[0]
    )
metabric["CANCER_STAGE"] = metabric["CANCER_STAGE"]

if "METASTASIS" not in metabric.columns:
    metabric["METASTASIS"] = None

metabric["SOURCE"] = "METABRIC"
final_cols = [
    "AGE", "TUMOR_SIZE", "NODES", "GRADE", "MENOPAUSE",
    "ER_STATUS", "PR_STATUS", "HER2_STATUS", "CELLULARITY",
    "METASTASIS", "SURVIVAL_MONTHS", "STATUS", "NPI",
    "SOURCE", "CANCER_STAGE"
]
for col in final_cols:
    if col not in metabric.columns:
        metabric[col] = None
metabric_final = metabric[final_cols]

print("METABRIC dataset loaded.")
print(metabric_final.head())


In [ ]:
# Cell 5: Load & preprocess SEER Dataset

seer = pd.read_csv(seer_path)
seer.columns = seer.columns.str.upper().str.strip()
rename_map = {
    "TUMOR SIZE": "TUMOR_SIZE",
    "N STAGE": "NODES",
    "6TH STAGE": "CANCER_STAGE",
    "ESTROGEN STATUS": "ER_STATUS",
    "PROGESTERONE STATUS": "PR_STATUS",
    "SURVIVAL MONTHS": "SURVIVAL_MONTHS",
    "A STAGE": "A_STAGE"
}
seer = seer.rename(columns={k: v for k, v in rename_map.items() if k in seer.columns})

seer["AGE"] = pd.to_numeric(seer["AGE"], errors="coerce")
seer["NODES"] = pd.to_numeric(seer["NODES"].astype(str).str.extract(r"(\d+)")[0], errors="coerce")
seer["TUMOR_SIZE"] = pd.to_numeric(seer["TUMOR_SIZE"], errors="coerce")
seer["STATUS"] = (
    seer["STATUS"].astype(str).str.strip().str.upper()
    .replace({"ALIVE": 0, "LIVING": 0, "DEAD": 1, "DECEASED": 1, "0": 0, "1": 1})
)
seer["STATUS"] = pd.to_numeric(seer["STATUS"], errors="coerce")
seer["STATUS"] = seer["STATUS"].fillna(seer["STATUS"].mode()[0]).astype(int)
seer["CANCER_STAGE"] = seer["CANCER_STAGE"].astype(str).str.strip()
seer["GRADE"] = seer["GRADE"].astype(str).str.strip().replace({'nan': None})
a_stage_mapping = {"REGIONAL": 0, "DISTANT": 1}
seer["METASTASIS"] = seer["A_STAGE"].astype(str).str.strip().str.upper().map(a_stage_mapping)
seer["METASTASIS"] = pd.to_numeric(seer["METASTASIS"], errors="coerce")

for col in ["MENOPAUSE", "HER2_STATUS", "CELLULARITY", "NPI"]:
    if col not in seer.columns:
        seer[col] = None

seer["MENOPAUSE"] = seer["MENOPAUSE"].fillna("Unknown").astype(str).str.title()
seer["MENOPAUSE"] = seer["MENOPAUSE"].where(seer["MENOPAUSE"].isin(["Pre", "Post"]), "Unknown")
seer["ER_STATUS"] = seer["ER_STATUS"].astype(object)
seer["PR_STATUS"] = seer["PR_STATUS"].astype(object)

seer["SOURCE"] = "SEER"
final_cols = [
    "AGE", "TUMOR_SIZE", "NODES", "GRADE", "MENOPAUSE",
    "ER_STATUS", "PR_STATUS", "HER2_STATUS", "CELLULARITY",
    "METASTASIS", "SURVIVAL_MONTHS", "STATUS", "NPI",
    "SOURCE", "CANCER_STAGE"
]

for col in final_cols:
    if col not in seer.columns:
        seer[col] = None

seer_final = seer[final_cols]
print("SEER dataset loaded.")
print(seer_final.head())

In [ ]:
# Cell 6: Merge all datasets

combined = pd.concat([metabric_final, seer_final, kaggle_final], ignore_index=True)
print("Final dataset:", combined.shape)
combined.head()


In [ ]:
# CELL 7: Dataset-Level Cleaning (AFTER MERGE)

TARGET = "CANCER_STAGE"
numeric_cols = ["AGE","TUMOR_SIZE","NODES","GRADE","SURVIVAL_MONTHS","NPI","CANCER_STAGE"]
for c in numeric_cols:
    if c in combined.columns:
        combined[c] = pd.to_numeric(combined[c], errors="coerce")
if "STATUS" in combined.columns:
    combined["STATUS"] = (
        combined["STATUS"]
        .astype(str).str.upper()
        .map({"ALIVE":0, "LIVING":0, "DEAD":1, "DECEASED":1, "0":0, "1":1})
    )
    combined["STATUS"] = pd.to_numeric(combined["STATUS"], errors="coerce")

combined = combined[
    (combined["AGE"].isna()) | ((combined["AGE"] > 0) & (combined["AGE"] < 120))
]
combined = combined[
    (combined["TUMOR_SIZE"].isna()) | (combined["TUMOR_SIZE"] >= 0)
]
combined = combined[
    (combined["SURVIVAL_MONTHS"].isna()) | (combined["SURVIVAL_MONTHS"] >= 0)
]

combined[TARGET] = pd.to_numeric(combined[TARGET], errors="coerce")

print("Dataset cleaned.")
print("Shape:", combined.shape)
print("Target distribution:\n", combined[TARGET].value_counts())
print("Number of rows with missing target:", combined[TARGET].isna().sum())
combined.head()

**Preprocessing**

In [ ]:
# Cell 8: Handling Missing Values

TARGET = "CANCER_STAGE"
numeric_cols = combined.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = combined.select_dtypes(include=["object", "category"]).columns.tolist()
if TARGET in categorical_cols:
    categorical_cols.remove(TARGET)

for col in numeric_cols:
    if col == TARGET:
        continue
    if combined[col].isnull().all():
        combined[col] = combined[col].fillna(0)
    else:
        combined[col] = combined[col].fillna(combined[col].median())

for col in categorical_cols:
    if combined[col].isnull().all() or combined[col].mode().empty:
        combined[col] = combined[col].fillna("Unknown")
    else:
        combined[col] = combined[col].fillna(combined[col].mode()[0])
    combined[col] = combined[col].astype("category")
def estimate_cancer_stage(row):
    try:
        if row["METASTASIS"] == 1:
            return 4
        if row["NODES"] >= 4 or row["TUMOR_SIZE"] >= 50:
            return 3
        if row["NODES"] >= 1 or row["TUMOR_SIZE"] >= 20:
            return 2
        return 1
    except:
        return None

missing_target_mask = combined[TARGET].isna()
combined.loc[missing_target_mask, TARGET] = combined.loc[missing_target_mask].apply(
    estimate_cancer_stage, axis=1
)

combined[TARGET] = pd.to_numeric(combined[TARGET], errors="coerce").astype("int64")

def calculate_npi(row):
    try:
        if pd.notna(row["TUMOR_SIZE"]) and pd.notna(row["NODES"]) and pd.notna(row["GRADE"]):
            tumor_score = 0.2 * row["TUMOR_SIZE"]
            if row["NODES"] == 0:
                node_score = 1
            elif 1 <= row["NODES"] <= 3:
                node_score = 2
            else:
                node_score = 3

            return tumor_score + node_score + row["GRADE"]
        else:
            return None
    except:
        return None
npi_missing_mask = combined["NPI"].isna()
combined.loc[npi_missing_mask, "NPI"] = combined.loc[npi_missing_mask].apply(calculate_npi, axis=1)
if combined["NPI"].isna().any():
    combined["NPI"] = combined["NPI"].fillna(combined["NPI"].median())
print("Remaining missing values:\n", combined.isna().sum())
print(f"\n{TARGET} distribution:\n", combined[TARGET].value_counts())
combined.head()

In [ ]:
# Cell 9: Remove Duplicates

combined = combined.drop_duplicates()
print("After dropping duplicates:", combined.shape)
print("Remaining missing values:\n", combined.isna().sum())
combined.head()

In [ ]:
# Cell 10: Feature Encoding + Numeric Cleaning
df = combined.copy()
junk_values = ["#", "Unknown", "unknown", "?", " ", "", "None", "NA", "N/A"]
df = df.replace(junk_values, pd.NA)

exclude_cols = ["CANCER_STAGE", "STATUS", "METASTASIS", "SOURCE"]
categorical_cols = [
    col for col in df.columns
    if (df[col].dtype == "object" or str(df[col].dtype).startswith("category"))
    and col not in exclude_cols
]

print("Categorical columns to encode:", categorical_cols)

le = LabelEncoder()
for col in categorical_cols:
    df[col] = df[col].astype(str)
    df[col] = le.fit_transform(df[col])

df = df.apply(pd.to_numeric, errors="coerce")
numeric_cols = df.select_dtypes(include="number").columns.tolist()
numeric_cols = [col for col in numeric_cols if col != "SOURCE"]

df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

print("Non-numeric columns left:", df.select_dtypes(exclude="number").columns.tolist())
print("Remaining NaN count:", df.drop(columns=["SOURCE"]).isna().sum().sum())

combined = df.copy()
print("Feature encoding amd numeric cleaning completed successfully.")
combined.head()


In [ ]:
# Cell 11: Min-Max Normalization

TARGET = "CANCER_STAGE"
normalize_cols = [
    "AGE",
    "TUMOR_SIZE",
    "NODES",
    "SURVIVAL_MONTHS",
    "NPI"
]
scaler = MinMaxScaler()
combined[normalize_cols] = scaler.fit_transform(combined[normalize_cols])
print("Min-Max normalization applied successfully.")
print("Normalized columns:", normalize_cols)
combined.head()


In [ ]:
# Cell 12: Data Balancing (SMOTE)

TARGET = "CANCER_STAGE"
X = combined.drop(columns=[TARGET, "SOURCE"])
y = combined[TARGET]

print("Original class distribution:")
print(Counter(y))
categorical_cols = [
    "GRADE",
    "MENOPAUSE",
    "ER_STATUS",
    "PR_STATUS",
    "HER2_STATUS",
    "CELLULARITY",
    "METASTASIS",
    "STATUS"
]
for col in X.columns:
    X[col] = pd.to_numeric(X[col], errors="coerce").fillna(0)
for col in categorical_cols:
    if col in X.columns:
        X[col] = X[col].astype(int)
categorical_indices = [
    X.columns.get_loc(col)
    for col in categorical_cols
    if col in X.columns
]

smote_nc = SMOTENC(
    categorical_features=categorical_indices,
    sampling_strategy="not majority",
    random_state=42,
    k_neighbors=5
)

X_resampled, y_resampled = smote_nc.fit_resample(X, y)

print("\nResampled class distribution:")
print(Counter(y_resampled))
X_resampled = pd.DataFrame(X_resampled, columns=X.columns)
y_resampled = pd.Series(y_resampled, name=TARGET)

balanced_data = pd.concat([X_resampled, y_resampled], axis=1)
print("\nBalanced dataset shape:", balanced_data.shape)
balanced_data.head()

In [ ]:
# Cell 13: Feature Selection

TARGET = "CANCER_STAGE"
X = balanced_data.drop(columns=[TARGET])
y = balanced_data[TARGET]
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X.select_dtypes(include=['category', 'object', 'bool']).columns.tolist()
var_thresh = VarianceThreshold(threshold=0.01)
X_var = var_thresh.fit_transform(X[numeric_cols])
selected_numeric_cols = [
    col for col, keep in zip(numeric_cols, var_thresh.get_support()) if keep
]
print("Numeric features after variance threshold:", selected_numeric_cols)

X_num_df = pd.DataFrame(X_var, columns=selected_numeric_cols)
corr_matrix = X_num_df.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

clinical_features = ['NODES', 'ER_STATUS']

to_drop_corr = [
    col for col in upper.columns
    if any(upper[col] > 0.9) and col not in clinical_features
]
X_num_df = X_num_df.drop(columns=to_drop_corr)

for col in clinical_features:
    if col not in X_num_df.columns and col in X.columns:
        X_num_df[col] = X[col]

print("Numeric features after correlation filter:", X_num_df.columns.tolist())

anova_selector = SelectKBest(score_func=f_classif, k='all')
anova_selector.fit(X_num_df, y)
anova_selected = [
    col for col, pval in zip(X_num_df.columns, anova_selector.pvalues_)
    if pval < 0.05
]
X_num_final = X_num_df[anova_selected]
print("Numeric features after ANOVA F-test:", X_num_final.columns.tolist())

if categorical_cols:
    X_cat = X[categorical_cols].apply(
        lambda col: col.cat.codes if str(col.dtype) == 'category' else col
    )
    chi_selector = SelectKBest(score_func=chi2, k='all')
    chi_selector.fit(X_cat, y)
    chi_selected = [
        col for col, pval in zip(X_cat.columns, chi_selector.pvalues_)
        if pval < 0.05
    ]
    X_cat_final = X_cat[chi_selected]
else:
    X_cat_final = pd.DataFrame()

print("Categorical features after Chi-square:", X_cat_final.columns.tolist())

X_fs = pd.concat([X_num_final, X_cat_final], axis=1)
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_fs, y)

importances = pd.Series(rf.feature_importances_, index=X_fs.columns)
tree_selected = importances[importances > 0.01].index.tolist()

for col in clinical_features:
    if col not in tree_selected and col in X_fs.columns:
        tree_selected.append(col)

X_tree_final = X_fs[tree_selected]
print("Features selected by tree importance:", tree_selected)

rfe_estimator = RandomForestClassifier(n_estimators=100, random_state=42)
n_features_to_select = min(12, X_tree_final.shape[1])
rfe_selector = RFE(
    estimator=rfe_estimator,
    n_features_to_select=n_features_to_select,
    step=1
)
rfe_selector.fit(X_tree_final, y)

rfe_selected_features = X_tree_final.columns[rfe_selector.support_].tolist()

for col in clinical_features:
    if col not in rfe_selected_features and col in X_tree_final.columns:
        rfe_selected_features.append(col)

X_final = X_tree_final[rfe_selected_features]

print("Final selected features after RFE:", rfe_selected_features)

final_dataset = pd.concat(
    [X_final, y.reset_index(drop=True)],
    axis=1
)
print("Final dataset shape:", final_dataset.shape)
final_dataset.head()


In [ ]:
# Cell 14: XAI (SHAP for Multi-class Cancer Stage – Full Balanced Dataset)

TARGET = "CANCER_STAGE"
X = balanced_data.drop(columns=[TARGET, "SOURCE"], errors="ignore")
y = balanced_data[TARGET]

print("X shape:", X.shape)
print("y shape:", y.shape)
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X, y)
print("Model trained successfully")

# SHAP TreeExplainer
explainer = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(X, check_additivity=False)
print("SHAP values computed for full dataset")
plt.figure(figsize=(6,4))
shap.summary_plot(
    shap_values,
    X,
    plot_type="bar",
    show=True,
    max_display=12
)

# Beeswarm plot
plt.figure(figsize=(10,6))
shap.summary_plot(shap_values, X, show=True)

sample_idx = 0
pred_class = rf_model.predict([X.iloc[sample_idx]])[0]
class_id = pred_class - 1
print(f"Predicted stage for sample {sample_idx}: {pred_class}")

# Waterfall plot for local explanation
plt.figure(figsize=(10,6))
shap.plots.waterfall(
    shap.Explanation(
        values=shap_values[class_id][sample_idx],
        base_values=explainer.expected_value[class_id],
        data=X.iloc[sample_idx],
        feature_names=X.columns
    )
)

In [ ]:
# Cell 14.1: XAI (SHAP for Multi-class Cancer Stage for random 500 samples)

TARGET = "CANCER_STAGE"

X = combined.drop(columns=[TARGET, "SOURCE"])
y = combined[TARGET]
rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X, y)
print("Model trained successfully")

# 500 random samples for faster XAI
X_sample = X.sample(n=500, random_state=42)

# SHAP TreeExplainer
explainer = shap.TreeExplainer(rf_model)

shap_values = explainer.shap_values(X_sample, check_additivity=False)
print("SHAP values computed")

# Bar plot (mean absolute SHAP)
plt.figure(figsize=(2,1))
shap.summary_plot(
    shap_values,
    X_sample,
    plot_type="bar",
    show=True,
    max_display=12
)

# Beeswarm plot
plt.figure(figsize=(2,1))
shap.summary_plot(shap_values, X_sample, show=True)
sample_idx = 0
pred_class = rf_model.predict([X_sample.iloc[sample_idx]])[0]
class_id = pred_class - 1
print(f"Predicted stage for sample {sample_idx}: {pred_class}")

# Waterfall plot
plt.figure(figsize=(2,1))
shap.plots.waterfall(
    shap.Explanation(
        values=shap_values[class_id][sample_idx],
        base_values=explainer.expected_value[class_id],
        data=X_sample.iloc[sample_idx],
        feature_names=X_sample.columns
    )
)

In [ ]:
# Cell 15: Hyperparameter Tuning (GridSearch)

TARGET = "CANCER_STAGE"

X = balanced_data.drop(columns=[TARGET, "SOURCE"], errors="ignore")
y = balanced_data[TARGET]
print("X shape:", X.shape)
print("y shape:", y.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Random Forest
rf = RandomForestClassifier(
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

# Conservative Grid
param_grid = {
    'n_estimators': [50, 75, 100],
    'max_depth': [3, 4, 5, 6],
    'min_samples_split': [8, 10, 12],
    'min_samples_leaf': [3, 4, 5],
    'max_features': ['sqrt', 'log2']
}

# GridSearchCV
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2
)

print("Fitting GridSearchCV")
grid_search.fit(X_train, y_train)

print("\n Best Hyperparameters:")
print(grid_search.best_params_)

best_rf = grid_search.best_estimator_
y_pred = best_rf.predict(X_test)

print("\nTest Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))


In [ ]:
# Cell 16: Results Visualization

# Feature Importance (Random Forest)
feature_importances = best_rf.feature_importances_
features = X.columns

plt.figure(figsize=(10,6))
sns.barplot(x=feature_importances, y=features, palette="viridis")
plt.title("Random Forest Feature Importance")
plt.xlabel("Importance")
plt.ylabel("Features")
plt.tight_layout()
plt.show()

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8,6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=best_rf.classes_)
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion Matrix on Test Set")
plt.show()

# Bar Plot (Mean Absolute SHAP)
plt.figure(figsize=(10,6))
shap.summary_plot(shap_values, X_sample, plot_type="bar", max_display=12, show=True)

# Beeswarm Plot
plt.figure(figsize=(10,6))
shap.summary_plot(shap_values, X_sample, show=True)

# SHAP Waterfall Plot
sample_idx = 0
pred_class = rf_model.predict([X_sample.iloc[sample_idx]])[0]
class_id = pred_class - 1

plt.figure(figsize=(10,6))
shap.plots.waterfall(
    shap.Explanation(
        values=shap_values[class_id][sample_idx],
        base_values=explainer.expected_value[class_id],
        data=X_sample.iloc[sample_idx],
        feature_names=X_sample.columns
    )
)


In [ ]:
# Cell 17: XGBoost

TARGET = "CANCER_STAGE"
X = balanced_data.drop(columns=[TARGET, "SOURCE"], errors="ignore")
y = balanced_data[TARGET]

print("X shape:", X.shape)
print("y shape:", y.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

xgb_model = xgb.XGBClassifier(
    n_estimators=40,
    max_depth=2,
    learning_rate=0.15,
    subsample=0.6,
    colsample_bytree=0.6,
    objective='multi:softmax',
    num_class=len(y.unique()),
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)
print("XGBoost trained successfully")
y_pred = xgb_model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print("\nTest Accuracy:", acc)
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

In [ ]:
# Cell 17.1: Visualization

# Feature Importance
xgb_importances = xgb_model.feature_importances_
features = X.columns
plt.figure(figsize=(10,6))
sns.barplot(x=xgb_importances, y=features, palette="magma")
plt.title("XGBoost Feature Importance")
plt.xlabel("Importance")
plt.ylabel("Features")
plt.tight_layout()
plt.show()

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8,6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=xgb_model.classes_)
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion Matrix on Test Set (XGBoost)")
plt.show()

# SHAP Feature Importance
X_sample = X.sample(n=500, random_state=42)
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_sample)

# Bar plot (mean absolute SHAP)
plt.figure(figsize=(10,6))
shap.summary_plot(shap_values, X_sample, plot_type="bar", max_display=12, show=True)

# Beeswarm plot
plt.figure(figsize=(10,6))
shap.summary_plot(shap_values, X_sample, show=True)

# SHAP Waterfall Plot (
sample_idx = 0
pred_class = xgb_model.predict([X_sample.iloc[sample_idx]])[0]
class_id = pred_class - 1
plt.figure(figsize=(10,6))
shap.plots.waterfall(
    shap.Explanation(
        values=shap_values[class_id][sample_idx],
        base_values=explainer.expected_value[class_id],
        data=X_sample.iloc[sample_idx],
        feature_names=X_sample.columns
    )
)
plt.show()


In [ ]:
# Cell 18: Logistic Regression

TARGET = "CANCER_STAGE"
X = balanced_data.drop(columns=[TARGET, "SOURCE"], errors="ignore")
y = balanced_data[TARGET]

print("X shape:", X.shape)
print("y shape:", y.shape)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Logistic Regression Classifier
logreg = LogisticRegression(
    multi_class='ovr',
    solver='lbfgs',
    max_iter=500,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)

logreg.fit(X_train, y_train)
print("Logistic Regression trained successfully")

y_pred = logreg.predict(X_test)

print("\nTest Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))


In [ ]:
# Cell 18.1: Visualization

# Feature Importance
coefficients = np.mean(np.abs(logreg.coef_), axis=0)
features = X.columns
plt.figure(figsize=(10,6))
sns.barplot(x=coefficients, y=features, palette="magma")
plt.title("Logistic Regression Feature Importance")
plt.xlabel("Coefficient Magnitude")
plt.ylabel("Features")
plt.tight_layout()
plt.show()

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8,6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=logreg.classes_)
disp.plot(cmap="Blues", values_format="d")
plt.title("Logistic Regression Confusion Matrix")
plt.show()

plt.figure(figsize=(10,6))
sns.barplot(x=coefficients, y=features, palette="magma")
plt.title("Logistic Regression Feature Importance")
plt.xlabel("Coefficient Magnitude")
plt.ylabel("Features")
plt.tight_layout()
plt.savefig("logreg_feature_importance.png", dpi=300)
plt.close()

plt.figure(figsize=(8,6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=logreg.classes_)
disp.plot(cmap="Blues", values_format="d")
plt.title("Logistic Regression Confusion Matrix")
plt.savefig("logreg_confusion_matrix.png", dpi=300)
plt.close()

print("Logistic Regression charts saved as PNG files")


In [ ]:
# Cell 19: SVM Classifier
TARGET = "CANCER_STAGE"

X = balanced_data.drop(columns=[TARGET, "SOURCE"], errors="ignore")
y = balanced_data[TARGET]

print("X shape:", X.shape)
print("y shape:", y.shape)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
# SVM Classifier
svm_model = SVC(
    kernel='rbf',
    C=5.0,
    gamma='scale',
    class_weight='balanced',
    random_state=42,
    probability=True
)
svm_model.fit(X_train, y_train)
print("SVM trained successfully")
y_pred = svm_model.predict(X_test)

print("\nTest Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))


In [ ]:
# Cell 19.1: Visualization

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8,6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=svm_model.classes_)
disp.plot(cmap="Blues", values_format="d")
plt.title("SVM Confusion Matrix")
plt.show()

# Permutation Feature Importance
result = permutation_importance(
    svm_model, X_test, y_test,
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

# Feature Importance
importances = result.importances_mean
features = X_test.columns
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10,6))
sns.barplot(x=importances[indices], y=features[indices], palette="coolwarm")
plt.title("SVM Feature Importance")
plt.xlabel("Importance")
plt.ylabel("Features")
plt.tight_layout()
plt.show()


In [ ]:
# Cell 20: LightGBM Classifier
TARGET = "CANCER_STAGE"

X = balanced_data.drop(columns=[TARGET, "SOURCE"], errors="ignore")
y = balanced_data[TARGET]

print("X shape:", X.shape)
print("y shape:", y.shape)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
# LightGBM Classifier
lgb_model = lgb.LGBMClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    num_leaves=31,
    objective='multiclass',
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

# Train model
lgb_model.fit(X_train, y_train)
print("LightGBM trained successfully")

# Evaluate on test set
y_pred = lgb_model.predict(X_test)

print("\nTest Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))


In [ ]:
# Cell 20.1: LightGBM Visualization

# Feature Importance
feature_importances = lgb_model.feature_importances_
features = X.columns

plt.figure(figsize=(10,6))
sns.barplot(x=feature_importances, y=features, palette="viridis")
plt.title("LightGBM Feature Importance")
plt.xlabel("Importance")
plt.ylabel("Features")
plt.tight_layout()
plt.show()

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8,6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=lgb_model.classes_)
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion Matrix on LightGBM")
plt.show()

# SHAP Global Feature Importance
X_sample = X.sample(n=500, random_state=42)
explainer = shap.TreeExplainer(lgb_model)
shap_values = explainer.shap_values(X_sample)

# Bar Plot (Mean Absolute SHAP)
plt.figure(figsize=(10,6))
shap.summary_plot(shap_values, X_sample, plot_type="bar", max_display=12, show=True)

# Beeswarm Plot
plt.figure(figsize=(10,6))
shap.summary_plot(shap_values, X_sample, show=True)

# SHAP Waterfall Plot
sample_idx = 0
pred_class = lgb_model.predict([X_sample.iloc[sample_idx]])[0]
class_id = pred_class - 1
plt.figure(figsize=(10,6))
shap.plots.waterfall(
    shap.Explanation(
        values=shap_values[class_id][sample_idx],
        base_values=explainer.expected_value[class_id],
        data=X_sample.iloc[sample_idx],
        feature_names=X_sample.columns
    )
)
plt.show()

In [ ]:
# Cell 20.2: Multiclass ROC and Precision-Recall Curves for LightGBM

import matplotlib.pyplot as plt
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score
import numpy as np

TARGET = "CANCER_STAGE"
X = balanced_data.drop(columns=[TARGET, "SOURCE"], errors="ignore")
y = balanced_data[TARGET]

# Train-test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

import lightgbm as lgb
lgb_model = lgb.LGBMClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    num_leaves=31,
    objective='multiclass',
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
lgb_model.fit(X_train, y_train)

classes = np.unique(y)
y_test_bin = label_binarize(y_test, classes=classes)
y_score = lgb_model.predict_proba(X_test)

# ROC Curve
plt.figure(figsize=(10,6))
for i, class_label in enumerate(classes):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_score[i][:, i] if isinstance(y_score, list) else y_score[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, lw=2, label=f"Class {class_label} (AUC = {roc_auc:.2f})")

plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Multiclass ROC Curve")
plt.legend(loc="lower right")
plt.show()

# Precision-Recall Curve

plt.figure(figsize=(10,6))
for i, class_label in enumerate(classes):
    precision, recall, _ = precision_recall_curve(y_test_bin[:, i], y_score[i][:, i] if isinstance(y_score, list) else y_score[:, i])
    ap = average_precision_score(y_test_bin[:, i], y_score[i][:, i] if isinstance(y_score, list) else y_score[:, i])
    plt.plot(recall, precision, lw=2, label=f"Class {class_label} (AP = {ap:.2f})")

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Multiclass Precision-Recall Curve")
plt.legend(loc="lower left")
plt.show()


In [ ]:
# Cell 21.0: Install CatBoost
!pip install catboost --quiet


In [ ]:
# Cell 21: CatBoost

from catboost import CatBoostClassifier

# Initialize CatBoost
cat_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.1,
    depth=6,
    loss_function='MultiClass',
    eval_metric='MultiClass',
    random_seed=42,
    verbose=50
)
# Train model
cat_model.fit(X_train, y_train, eval_set=(X_test, y_test))
y_pred = cat_model.predict(X_test)

# Test accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy:.3f}\n")

# Classification report
print("Classification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
# Cell 21.1: Visualizatiom

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=[0,1,2,3,4], yticklabels=[0,1,2,3,4])
plt.title("CatBoost Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# Feature Importance
plt.figure(figsize=(8,6))
feature_importances = cat_model.get_feature_importance()
sns.barplot(x=feature_importances, y=X_train.columns)
plt.title("CatBoost Feature Importance")
plt.xlabel("Importance")
plt.ylabel("Features")
plt.show()


In [ ]:
# Cell 22: AdaBoost

# Initialize AdaBoost
base_estimator = DecisionTreeClassifier(max_depth=3, random_state=42)
ada_model = AdaBoostClassifier(
    estimator=base_estimator,
    n_estimators=500,
    learning_rate=0.1,
    random_state=42
)

# Train AdaBoost model
ada_model.fit(X_train, y_train)
y_pred_ada = ada_model.predict(X_test)

accuracy_ada = accuracy_score(y_test, y_pred_ada)
print(f"Test Accuracy: {accuracy_ada:.3f}\n")

print("Classification Report:")
print(classification_report(y_test, y_pred_ada))

print("Confusion Matrix:")
cm_ada = confusion_matrix(y_test, y_pred_ada)
print(cm_ada)


In [ ]:
# Cell 22.1: Visualization

# Confusion Matrix
plt.figure(figsize=(6,5))
sns.heatmap(cm_ada, annot=True, fmt='d', cmap='Oranges', xticklabels=[0,1,2,3,4], yticklabels=[0,1,2,3,4])
plt.title("AdaBoost Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# Feature Importance
plt.figure(figsize=(8,6))
feature_importances_ada = ada_model.feature_importances_
sns.barplot(x=feature_importances_ada, y=X_train.columns)
plt.title("AdaBoost Feature Importance")
plt.xlabel("Importance")
plt.ylabel("Features")
plt.show()


In [ ]:
# Cell 23: Decision Tree

# Initialize Decision Tree
dt_model = DecisionTreeClassifier(max_depth=None, random_state=42)

# Train the model
dt_model.fit(X_train, y_train)
y_pred_dt = dt_model.predict(X_test)

# Evaluation
accuracy_dt = accuracy_score(y_test, y_pred_dt)
print(f"Test Accuracy: {accuracy_dt:.3f}\n")

print("Classification Report:")
print(classification_report(y_test, y_pred_dt))

print("Confusion Matrix:")
cm_dt = confusion_matrix(y_test, y_pred_dt)
print(cm_dt)


In [ ]:
# Cell 23.1: Visualization

plt.figure(figsize=(20,10))
plot_tree(
    dt_model,
    feature_names=X.columns,
    class_names=[str(i) for i in sorted(y.unique())],
    filled=True,
    rounded=True,
    fontsize=10
)
# Feature Importance
import pandas as pd

feature_importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': dt_model.feature_importances_
}).sort_values(by='importance', ascending=False)

plt.figure(figsize=(12,6))
plt.barh(feature_importance_df['feature'], feature_importance_df['importance'], color='skyblue')
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Decision Tree Feature Importances")
plt.gca().invert_yaxis()
plt.show()

plt.title("Decision Tree Visualization")
plt.show()


In [ ]:
# Cell 24: LightGBM Hyperparameter Tuning

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

lgbm_base = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=len(y.unique()),
    random_state=42,
    verbose=-1
)
param_grid_fast = {
    'n_estimators': [200, 300],
    'learning_rate': [0.05, 0.1],
    'num_leaves': [31, 50],
    'max_depth': [10, -1],
    'min_child_samples': [30, 50],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'reg_alpha': [0.0, 0.1],
    'reg_lambda': [0.0, 0.1]
}

grid_search_lgbm_fast = GridSearchCV(
    estimator=lgbm_base,
    param_grid=param_grid_fast,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search_lgbm_fast.fit(X_train, y_train)

best_lgbm_model_fast = grid_search_lgbm_fast.best_estimator_

print("Best Hyperparameters:")
print(grid_search_lgbm_fast.best_params_)

y_pred_lgbm_fast = best_lgbm_model_fast.predict(X_test)

accuracy_lgbm_fast = accuracy_score(y_test, y_pred_lgbm_fast)
print(f"\n Tuned LightGBM Test Accuracy: {accuracy_lgbm_fast:.3f}")

print("\n Classification Report:")
print(classification_report(y_test, y_pred_lgbm_fast))

print("Confusion Matrix:")
cm_lgbm_fast = confusion_matrix(y_test, y_pred_lgbm_fast)
print(cm_lgbm_fast)

train_acc = best_lgbm_model_fast.score(X_train, y_train)
print(f"\n Train Accuracy: {train_acc:.3f}")
print(f" Test Accuracy: {accuracy_lgbm_fast:.3f}")


In [ ]:
# Cell 24.1: Visualizations

# Accuracy
models = ['LightGBM (Fast)']
accuracies = [accuracy_lgbm_fast]

plt.figure(figsize=(6,4))
plt.bar(models, accuracies, color=['skyblue'])
plt.ylabel("Accuracy")
plt.title("Accuracy of LightGBM")
plt.ylim(0.9, 1.0)

for i, v in enumerate(accuracies):
    plt.text(i, v + 0.002, f"{v:.3f}", ha='center', fontweight='bold')

plt.show()

# Confusion Matrix
plt.figure(figsize=(6,6))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_lgbm_fast,
    display_labels=sorted(y.unique())
)
disp.plot(cmap='Blues', values_format='d')
plt.title("Confusion Matrix: LightGBM")
plt.show()

# Feature Importance
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': best_lgbm_model_fast.feature_importances_
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(8,5))
plt.barh(feature_importance['Feature'], feature_importance['Importance'], color='skyblue')
plt.xlabel("Importance Score")
plt.title("Feature Importance: LightGBM")
plt.gca().invert_yaxis()
plt.show()


In [ ]:
# Cell 25: Gradient Boosting Classifier (Sklearn)

# Initialize Gradient Boosting Classifier
gb_model = GradientBoostingClassifier(
    n_estimators=500,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

# Train the model
gb_model.fit(X_train, y_train)
y_pred_gb = gb_model.predict(X_test)

# Evaluation
accuracy_gb = accuracy_score(y_test, y_pred_gb)
print(f" Test Accuracy: {accuracy_gb:.3f}\n")

print("Classification Report:")
print(classification_report(y_test, y_pred_gb))

cm_gb = confusion_matrix(y_test, y_pred_gb)
print("Confusion Matrix:")
print(cm_gb)


In [ ]:
# Cell 25.1: Visualizations

# Accuracy
models = ['Gradient Boosting']
accuracies = [accuracy_gb]

plt.figure(figsize=(5,4))
plt.bar(models, accuracies, color='skyblue')
plt.ylim(0,1)
plt.ylabel("Accuracy")
plt.title("Test Accuracy: Gradient Boosting")

for i, v in enumerate(accuracies):
    plt.text(i, v + 0.01, f"{v:.3f}", ha='center', fontweight='bold')

plt.show()

# Confusion Matrix
plt.figure(figsize=(6,6))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_gb,
    display_labels=sorted(y.unique())
)
disp.plot(cmap='Blues', values_format='d')
plt.title("Confusion Matrix: Gradient Boosting")
plt.show()

# Feature Importance
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': gb_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(8,5))
plt.barh(feature_importance['Feature'], feature_importance['Importance'], color='lightgreen')
plt.xlabel("Importance Score")
plt.title("Feature Importance: Gradient Boosting")
plt.gca().invert_yaxis()
plt.show()


In [ ]:
# Cell 26: Random Forest Model (Base Model)
TARGET = "CANCER_STAGE"
X = balanced_data.drop(columns=[TARGET, "SOURCE"], errors="ignore")
y = balanced_data[TARGET]

print("X shape:", X.shape)
print("y shape:", y.shape)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
rf_baseline = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)
rf_baseline.fit(X_train, y_train)

y_pred_baseline = rf_baseline.predict(X_test)

test_accuracy = accuracy_score(y_test, y_pred_baseline)
print("\n Test Accuracy (Baseline RF):", test_accuracy)

print("\n Classification Report:")
print(classification_report(y_test, y_pred_baseline))

print("\n Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_baseline))



In [ ]:
# Cell 26.1: Visualizations

# Accuracy
y_pred = rf_baseline.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print("\n Test Accuracy (Baseline):", accuracy)

print("\n Classification Report:")
print(classification_report(y_test, y_pred))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8,6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=rf_baseline.classes_)
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion Matrix of Random Forest")
plt.show()

# Feature Importance
importances = rf_baseline.feature_importances_
features = X.columns
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10,6))
sns.barplot(x=importances[indices], y=features[indices], palette="viridis")
plt.title("Feature Importance of  Random Forest")
plt.xlabel("Importance")
plt.ylabel("Features")
plt.tight_layout()
plt.show()

# SHAP Analysis
explainer = shap.TreeExplainer(rf_baseline)
shap_values = explainer.shap_values(X_test)

# SHAP summary plot and dot plot
plt.figure()
shap.summary_plot(shap_values, X_test, plot_type="bar", show=True)
plt.figure()
shap.summary_plot(shap_values, X_test, show=True)


In [ ]:
# Cell 27: Extra Trees Classifier (Sklearn)

# Initialize Extra Trees Classifier
et_model = ExtraTreesClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'
)

# Train the model
et_model.fit(X_train, y_train)
y_pred_et = et_model.predict(X_test)

accuracy_et = accuracy_score(y_test, y_pred_et)
print(f" Test Accuracy (Extra Trees): {accuracy_et:.3f}\n")

print(" Classification Report:")
print(classification_report(y_test, y_pred_et))

cm_et = confusion_matrix(y_test, y_pred_et)
print(" Confusion Matrix:")
print(cm_et)

# Feature Importance
import pandas as pd
import matplotlib.pyplot as plt

feature_importance_et = pd.DataFrame({
    'Feature': X.columns,
    'Importance': et_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(8,5))
plt.barh(feature_importance_et['Feature'], feature_importance_et['Importance'], color='purple')
plt.xlabel("Importance Score")
plt.title("Feature Importance of Extra Trees Classifier")
plt.gca().invert_yaxis()
plt.show()


**Deep Learning**

In [ ]:
# Cell 28: Train-test split

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

TARGET = "CANCER_STAGE"
X = final_dataset.drop(columns=[TARGET])
y = final_dataset[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)

y_train_cat = to_categorical(y_train_enc)
y_test_cat  = to_categorical(y_test_enc)

print("y_train_cat shape:", y_train_cat.shape)
print("y_test_cat shape :", y_test_cat.shape)


In [ ]:
# Cell 29: Deep learning model MLP

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

ann = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.5),

    Dense(32, activation='relu'),
    Dropout(0.4),

    Dense(y_train_cat.shape[1], activation='softmax')
])

ann.compile(
    optimizer=Adam(learning_rate=0.0003),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history_ann = ann.fit(
    X_train, y_train_cat,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

ann.evaluate(X_test, y_test_cat)

test_loss, test_acc = ann.evaluate(X_test, y_test_cat, verbose=0)
print(f"Test Accuracy: {test_acc:.4f}")

from sklearn.metrics import classification_report, precision_score, recall_score, f1_score
import numpy as np

y_pred_probs = ann.predict(X_test)

y_pred_classes = np.argmax(y_pred_probs, axis=1)
y_true_classes = np.argmax(y_test_cat, axis=1)

# Classification report
print(classification_report(y_true_classes, y_pred_classes, digits=4))

# Macro average
precision_macro = precision_score(y_true_classes, y_pred_classes, average='macro')
recall_macro = recall_score(y_true_classes, y_pred_classes, average='macro')
f1_macro = f1_score(y_true_classes, y_pred_classes, average='macro')

print(f"Precision (Macro): {precision_macro:.4f}")
print(f"Recall (Macro):    {recall_macro:.4f}")
print(f"F1 Score (Macro):  {f1_macro:.4f}")



In [ ]:
# Accuracy Plot
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.legend()
plt.show()

# Loss Plot
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.legend()
plt.show()

In [ ]:
# Cell 30: CNN

import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Conv1D, MaxPooling1D, Flatten
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt

TARGET = "CANCER_STAGE"

X = final_dataset.drop(columns=[TARGET])
y = final_dataset[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)

y_train_cat = to_categorical(y_train_enc)
y_test_cat  = to_categorical(y_test_enc)


X_train_cnn = np.expand_dims(X_train.values, axis=2)
X_test_cnn  = np.expand_dims(X_test.values, axis=2)

cnn = Sequential([
    Conv1D(16, kernel_size=2, activation='relu', input_shape=(X_train_cnn.shape[1], 1)),
    MaxPooling1D(pool_size=2),
    Dropout(0.5),

    Conv1D(8, kernel_size=2, activation='relu'),
    MaxPooling1D(pool_size=2),
    Dropout(0.4),

    Flatten(),
    Dense(y_train_cat.shape[1], activation='softmax')
])

cnn.compile(
    optimizer=Adam(learning_rate=0.0003),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train
history_cnn = cnn.fit(
    X_train_cnn, y_train_cat,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)],
    verbose=1
)
# Accuracy
test_loss, test_acc = cnn.evaluate(X_test_cnn, y_test_cat, verbose=0)
print(f"CNN Test Accuracy: {test_acc:.4f}")


y_pred_probs = cnn.predict(X_test_cnn)
y_pred_classes = np.argmax(y_pred_probs, axis=1)
y_true_classes = np.argmax(y_test_cat, axis=1)

# Classification
print("\nCNN Classification Report:\n")
print(classification_report(y_true_classes, y_pred_classes, digits=4))

# Macro precision, recall, F1
precision_macro = precision_score(y_true_classes, y_pred_classes, average='macro')
recall_macro = recall_score(y_true_classes, y_pred_classes, average='macro')
f1_macro = f1_score(y_true_classes, y_pred_classes, average='macro')

print(f"CNN Precision (Macro): {precision_macro:.4f}")
print(f"CNN Recall (Macro):    {recall_macro:.4f}")
print(f"CNN F1 Score (Macro):  {f1_macro:.4f}")


# Plot Training & Validation Curves

# Accuracy Curve
plt.figure()
plt.plot(history_cnn.history['accuracy'], label='Training Accuracy')
plt.plot(history_cnn.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('CNN Training vs Validation Accuracy')
plt.legend()
plt.grid(True)
plt.show()

# Loss Curve
plt.figure()
plt.plot(history_cnn.history['loss'], label='Training Loss')
plt.plot(history_cnn.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('CNN Training vs Validation Loss')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Cell 31: KNN

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

X = final_dataset.drop(columns=["CANCER_STAGE"])
y = final_dataset["CANCER_STAGE"]

le = LabelEncoder()
y_enc = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=101, stratify=y_enc
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

knn = KNeighborsClassifier(n_neighbors=15, weights='uniform')
knn.fit(X_train_scaled, y_train)

y_pred = knn.predict(X_test_scaled)
acc = accuracy_score(y_test, y_pred)
print(f"KNN Accuracy: {acc:.4f}")

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)

from sklearn.metrics import classification_report, precision_score, recall_score, f1_score, confusion_matrix

acc = accuracy_score(y_test, y_pred)
print(f"KNN Accuracy: {acc:.4f}")

# Classification

print("\nKNN Classification Report:\n")
print(classification_report(y_test, y_pred, digits=4))

# Macro Precision, Recall, F1

precision_macro = precision_score(y_test, y_pred, average='macro')
recall_macro = recall_score(y_test, y_pred, average='macro')
f1_macro = f1_score(y_test, y_pred, average='macro')

print(f"KNN Precision (Macro): {precision_macro:.4f}")
print(f"KNN Recall (Macro):    {recall_macro:.4f}")
print(f"KNN F1 Score (Macro):  {f1_macro:.4f}")


In [ ]:
# Cell 32: LSTM

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

TARGET = "CANCER_STAGE"
X = final_dataset.drop(columns=[TARGET])
y = final_dataset[TARGET]

le = LabelEncoder()
y_enc = le.fit_transform(y)
y_cat = to_categorical(y_enc)

X_train, X_test, y_train, y_test = train_test_split(
    X.values, y_cat, test_size=0.2, random_state=42, stratify=y_enc
)

X_train_noisy = X_train + np.random.normal(0, 0.1, X_train.shape)
X_test_noisy  = X_test  + np.random.normal(0, 0.1, X_test.shape)

X_train_lstm = np.expand_dims(X_train_noisy, axis=1)
X_test_lstm  = np.expand_dims(X_test_noisy, axis=1)

lstm_model = Sequential([
    LSTM(8, activation='tanh', input_shape=(X_train_lstm.shape[1], X_train_lstm.shape[2])),
    Dropout(0.7),
    Dense(y_cat.shape[1], activation='softmax')
])

lstm_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_lstm = lstm_model.fit(
    X_train_lstm, y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=8,
    verbose=1
)

test_loss, test_acc = lstm_model.evaluate(X_test_lstm, y_test, verbose=0)
print(f"LSTM Test Accuracy: {test_acc:.4f}")
import numpy as np
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score

y_pred_probs = lstm_model.predict(X_test_lstm)

y_pred_classes = np.argmax(y_pred_probs, axis=1)
y_true_classes = np.argmax(y_test, axis=1)

# Classification Report

print("\nLSTM Classification Report:\n")
print(classification_report(y_true_classes, y_pred_classes, digits=4))

# Macro Precision, Recall, F1
precision_macro = precision_score(y_true_classes, y_pred_classes, average='macro')
recall_macro = recall_score(y_true_classes, y_pred_classes, average='macro')
f1_macro = f1_score(y_true_classes, y_pred_classes, average='macro')

print(f"LSTM Precision (Macro): {precision_macro:.4f}")
print(f"LSTM Recall (Macro):    {recall_macro:.4f}")
print(f"LSTM F1 Score (Macro):  {f1_macro:.4f}")

**Ensemble Model**

In [ ]:
#Cell 33: LightGBM + CatBoost

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize
lgb_model = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, random_state=42)
cat_model = CatBoostClassifier(iterations=500, learning_rate=0.05, verbose=0, random_state=42)


ensemble_model = VotingClassifier(
    estimators=[('lgb', lgb_model), ('cat', cat_model)],
    voting='soft',
    weights=[1, 1]
)

ensemble_model.fit(X_train, y_train)
y_pred = ensemble_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print("Soft Voting Ensemble Accuracy(lightgbm+catboost):", accuracy)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize individual models
lgb_model = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, random_state=42)
cat_model = CatBoostClassifier(iterations=500, learning_rate=0.05, verbose=0, random_state=42)

ensemble_model = VotingClassifier(
    estimators=[('lgb', lgb_model), ('cat', cat_model)],
    voting='soft',
    weights=[1, 1]
)

ensemble_model.fit(X_train, y_train)
y_pred = ensemble_model.predict(X_test)

# Evaluation Metrics
accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred, digits=4)

print("Soft Voting Ensemble Accuracy (LightGBM + CatBoost):", accuracy)
print("\nConfusion Matrix:\n", conf_matrix)
print("\nClassification Report:\n", report)

In [ ]:
# Cell 33.1: Visualization

conf_matrix = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title("Confusion Matrix ")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

print("\nClassification Report:\n", classification_report(y_test, y_pred, digits=4))
print("\nAccuracy:", accuracy_score(y_test, y_pred))
features = X.columns

lgb_model.fit(X_train, y_train)
cat_model.fit(X_train, y_train, verbose=0)

# Feature importances
lgb_importances = lgb_model.feature_importances_
cat_importances = cat_model.get_feature_importance()

combined_importances = (lgb_importances + cat_importances) / 2

fi_df = pd.DataFrame({
    'Feature': features,
    'Importance': combined_importances
}).sort_values(by='Importance', ascending=True)

# Plot combined feature importance
plt.figure(figsize=(8,6))
sns.barplot(x='Importance', y='Feature', data=fi_df, palette="coolwarm")
plt.title("Feature Importance (LightGBM + CatBoost)")
plt.xlabel("Average Importance")
plt.ylabel("Features")
plt.show()

In [ ]:

classes = np.unique(y_test)
lb = LabelBinarizer()
y_test_binarized = lb.fit_transform(y_test)

y_score = ensemble_model.predict_proba(X_test)


plt.figure(figsize=(8,6))
for i, class_label in enumerate(classes):
    fpr, tpr, _ = roc_curve(y_test_binarized[:, i], y_score[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, lw=2, label=f'Class {class_label} (AUC = {roc_auc:.2f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Multiclass ROC Curve')
plt.legend(loc="lower right")
plt.show()

# Precision-Recall Curve
plt.figure(figsize=(8,6))
for i, class_label in enumerate(classes):
    precision, recall, _ = precision_recall_curve(y_test_binarized[:, i], y_score[:, i])
    avg_precision = average_precision_score(y_test_binarized[:, i], y_score[:, i])
    plt.plot(recall, precision, lw=2, label=f'Class {class_label} (AP = {avg_precision:.2f})')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Multiclass Precision-Recall Curve')
plt.legend(loc="lower left")
plt.show()

In [ ]:
explainer_lgb = shap.TreeExplainer(lgb_model)
explainer_cat = shap.TreeExplainer(cat_model)

shap_values_lgb = explainer_lgb.shap_values(X_test)
shap_values_cat = explainer_cat.shap_values(X_test)


def convert_shap(shap_values):
    if isinstance(shap_values, list):
        return np.mean(np.array(shap_values), axis=0)
    else:
        return shap_values

shap_lgb_matrix = convert_shap(shap_values_lgb)
shap_cat_matrix = convert_shap(shap_values_cat)


shap_combined_matrix = (shap_lgb_matrix + shap_cat_matrix) / 2

# SHAP Bar Plot
plt.figure(figsize=(10,6))
shap.summary_plot(shap_combined_matrix, X_test, plot_type="bar", show=True)

# SHAP Summary (Beeswarm) Plot (Combined)
plt.figure(figsize=(10,6))
shap.summary_plot(shap_combined_matrix, X_test, show=True)

In [ ]:
# Import libraries(lightgbm + Catboost + Extra trees Classifier)

try:
    from catboost import CatBoostClassifier
    cat_installed = True
except ImportError:
    cat_installed = False
    print("CatBoost is not installed. Install it with !pip install catboost to run this ensemble.")


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize the models
lgb_model = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, random_state=42)
extra_model = ExtraTreesClassifier(n_estimators=500, random_state=42)

if cat_installed:
    cat_model = CatBoostClassifier(iterations=500, learning_rate=0.05, verbose=0, random_state=42)
    estimators = [('lgb', lgb_model), ('cat', cat_model), ('extra', extra_model)]
else:
    estimators = [('lgb', lgb_model), ('extra', extra_model)]

# Create Soft Voting Ensemble
ensemble_model = VotingClassifier(
    estimators=estimators,
    voting='soft',
    weights=[1, 1, 1]
)

# Train the ensemble
ensemble_model.fit(X_train, y_train)
y_pred = ensemble_model.predict(X_test)

# Evaluation Metrics
accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred, digits=4)

print("Soft Voting Ensemble Accuracy(lightgbm + Catboost + Extra trees Classifier):", accuracy)
print("\nConfusion Matrix:\n", conf_matrix)
print("\nClassification Report:\n", report)

try:
    from catboost import CatBoostClassifier
    cat_installed = True
except ImportError:
    cat_installed = False
    print("CatBoost is not installed. Install it with !pip install catboost to run this ensemble.")


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

lgb_model = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, random_state=42)
extra_model = ExtraTreesClassifier(n_estimators=500, random_state=42)

if cat_installed:
    cat_model = CatBoostClassifier(iterations=500, learning_rate=0.05, verbose=0, random_state=42)
    estimators = [('lgb', lgb_model), ('cat', cat_model), ('extra', extra_model)]
else:
    estimators = [('lgb', lgb_model), ('extra', extra_model)]

# Soft Voting Ensemble
ensemble_model = VotingClassifier(
    estimators=estimators,
    voting='soft',
    weights=[1, 1, 1]
)

# Train
ensemble_model.fit(X_train, y_train)

# Predict
y_pred = ensemble_model.predict(X_test)

# Metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

conf_matrix = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred, digits=4)

print("Soft Voting Ensemble Accuracy(lightgbm + Catboost + Extra trees Classifier):", accuracy)
print("Weighted Precision:", precision)
print("Weighted Recall:", recall)
print("Weighted F1-Score:", f1)

print("\nConfusion Matrix:\n", conf_matrix)
print("\nFull Classification Report:\n", report)

In [ ]:
# Imports(CatBoost + Random Forest + Decision Tree)


try:
    from catboost import CatBoostClassifier
    cat_installed = True
except ImportError:
    cat_installed = False
    print("CatBoost not installed. Will run ensemble without it.")


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize models
lgb_model = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, random_state=42)
extra_model = ExtraTreesClassifier(n_estimators=500, random_state=42)

if cat_installed:
    cat_model = CatBoostClassifier(iterations=500, learning_rate=0.05, verbose=0, random_state=42)
    estimators = [('lgb', lgb_model), ('cat', cat_model), ('extra', extra_model)]
else:
    estimators = [('lgb', lgb_model), ('extra', extra_model)]

# Create soft voting ensemble
ensemble_model = VotingClassifier(
    estimators=estimators,
    voting='soft',
    weights=[1, 1, 1]
)

# Train ensemble
ensemble_model.fit(X_train, y_train)
y_pred = ensemble_model.predict(X_test)

# Evaluate
accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred, digits=4)

print("Ensemble Accuracy(CatBoost + Random Forest + Decision Tree):", accuracy)
print("\nConfusion Matrix:\n", conf_matrix)
print("\nClassification Report:\n", report)

In [ ]:
# Imports(CatBoost + Random Forest + Decision Tree)


# CatBoost import (optional)
try:
    from catboost import CatBoostClassifier
    cat_installed = True
except ImportError:
    cat_installed = False
    print("CatBoost is not installed. Install it using !pip install catboost")

# Split dataset (assume X, y are defined)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize individual models
rf_model = RandomForestClassifier(n_estimators=500, random_state=42)
dt_model = DecisionTreeClassifier(random_state=42)

if cat_installed:
    cat_model = CatBoostClassifier(iterations=500, learning_rate=0.05, verbose=0, random_state=42)
    estimators = [('cat', cat_model), ('rf', rf_model), ('dt', dt_model)]
else:
    estimators = [('rf', rf_model), ('dt', dt_model)]

# Soft Voting Ensemble
ensemble_model = VotingClassifier(
    estimators=estimators,
    voting='soft',       # soft voting uses probabilities
    weights=[1, 1, 1]    # equal weights
)

# Train ensemble
ensemble_model.fit(X_train, y_train)

# Predictions
y_pred = ensemble_model.predict(X_test)

# Evaluate
accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred, digits=4)

print("Ensemble Accuracy(CatBoost + Random Forest + Decision Tree):", accuracy)
print("\nConfusion Matrix:\n", conf_matrix)
print("\nClassification Report:\n", report)

In [ ]:
# Imports(CatBoost + Extra Trees Classifier + Decision Tree)


try:
    from catboost import CatBoostClassifier
    cat_installed = True
except ImportError:
    cat_installed = False
    print("CatBoost not installed. Install it using !pip install catboost")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize models
extra_model = ExtraTreesClassifier(n_estimators=500, random_state=42)
dt_model = DecisionTreeClassifier(random_state=42)

if cat_installed:
    cat_model = CatBoostClassifier(iterations=500, learning_rate=0.05, verbose=0, random_state=42)
    estimators = [('cat', cat_model), ('extra', extra_model), ('dt', dt_model)]
else:
    estimators = [('extra', extra_model), ('dt', dt_model)]

# Soft Voting Ensemble
ensemble_model = VotingClassifier(
    estimators=estimators,
    voting='soft',
    weights=[1, 1, 1]
)

ensemble_model.fit(X_train, y_train)

y_pred = ensemble_model.predict(X_test)

# Evaluation
accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred, digits=4)

print("Soft Voting Ensemble Accuracy(CatBoost + Extra Trees Classifier + Decision Tree):", accuracy)
print("\nConfusion Matrix:\n", conf_matrix)
print("\nClassification Report:\n", report)